# Homework 3 у Colab

Поклади офіційні `train.csv`, `test.csv` і `sample_submission.csv` у `MyDrive/AI_lab_3/data` або зміни шлях у клітинці налаштувань. Можна почати з CPU. Перед XLM-R вибери GPU через **Runtime → Change runtime type**. Після зміни середовища повтори налаштування. Спліт і результати зберігаються на Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import subprocess
import sys

ROOT = Path("/content/drive/MyDrive/AI_lab_3")
DATA = ROOT / "data"
RUNS = ROOT / "runs"
SPLIT = RUNS / "split.json"
REPO = Path("/content/AI_lab_3")
CODE_REF = "5d2c4ed24f35422508bce870c618f1c1f224a6a8"
TEXT = "text"
GROUP = None  # Set to the true source-document ID column when available.
LANGUAGE = None
DOMAIN = None

missing = [name for name in ("train.csv", "test.csv", "sample_submission.csv")
           if not (DATA / name).is_file()]
if missing:
    raise FileNotFoundError(f"Copy {missing} to {DATA}")

def call(*args, **kwargs):
    subprocess.run(args, check=True, **kwargs)

if not REPO.exists():
    call("git", "clone", "https://github.com/yelgor/AI_lab_3.git", str(REPO))
call("git", "checkout", "--detach", CODE_REF, cwd=REPO)
call(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", cwd=REPO)
RUNS.mkdir(parents=True, exist_ok=True)

def lab(*args):
    call(sys.executable, "lab.py", "--data", str(DATA), "--split", str(SPLIT),
         *map(str, args), cwd=REPO)

def run(name, command, *args):
    out = RUNS / name
    if (out / "metrics.json").is_file() or (name == "eda" and (out / "examples.csv").is_file()):
        print(f"Using completed run {out}")
        return out
    if out.exists():
        suffix = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
        shutil.move(str(out), str(RUNS / f"{name}-incomplete-{suffix}"))
    lab(command, *args, "--out", out)
    return out

import csv
with (DATA / "train.csv").open(encoding="utf-8-sig", newline="") as stream:
    print("Train columns:", next(csv.reader(stream)))

Після перевірки назв колонок задай `GROUP`, `LANGUAGE` та `DOMAIN` у клітинці налаштувань, якщо такі колонки є. Якщо вже маєш локальний `split.json`, скопіюй його в `runs/` перед цією клітинкою, щоб зберегти порівнюваність.

In [ ]:
if not SPLIT.exists():
    group = ["--group", GROUP] if GROUP else []
    lab("prepare", "--text", TEXT, "--sample", 30000, *group)

manifest = json.loads(SPLIT.read_text())
if (manifest["text"], manifest["group"]) != (TEXT, GROUP):
    raise ValueError("Set TEXT and GROUP to match the existing split.")

eda_args = []
if LANGUAGE:
    eda_args.extend(["--language", LANGUAGE])
if DOMAIN:
    eda_args.extend(["--domain", DOMAIN])
run("eda", "eda", *eda_args)

Перший CPU baseline використовує ту саму фіксовану підвибірку, що й енкодер, і до 2 000 символів на документ для TF‑IDF. Стилометрія порівнюється з цим baseline на повному спільному validation. Підвибірка й ліміт довжини зменшують кількість кандидатів у словник ще до застосування `max_features`.

In [ ]:
baseline = RUNS / "baseline-30k"
if not (baseline / "metrics.json").is_file():
    if baseline.exists():
        suffix = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
        shutil.move(str(baseline), str(RUNS / f"baseline-30k-incomplete-{suffix}"))

    # Limit the fitting partition using IDs already saved in the shared manifest.
    cpu_code = """import lab
original_partitions = lab.partitions

def subset_partitions(args):
    text, _, val, sample = original_partitions(args)
    return text, sample, val, sample

lab.partitions = subset_partitions
lab.main()
"""
    call(sys.executable, "-c", cpu_code, "--data", str(DATA), "--split", str(SPLIT),
         "baseline", "--features", "40000", "--max-chars", "2000", "--out", str(baseline),
         cwd=REPO)
cpu_models = [baseline / "tfidf", baseline / "stylometry"]
first = run("first-colab", "select", *cpu_models)
print(first / "submission_1.csv")

Запускай варіанти XLM-R по одному. Після кожного завершеного запуску метрики та прогнози залишаються на Drive. Останню клітинку можна виконати після будь-якого завершеного варіанта.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before running XLM-R.")
print(torch.cuda.get_device_name(0))
call(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-gpu.txt", cwd=REPO)
GPU_ARGS = ["--batch", "4", "--accumulation", "8", "--length", "256"]

In [ ]:
run("colab-head", "encoder", *GPU_ARGS)

In [ ]:
run("colab-head-tail", "encoder", *GPU_ARGS, "--strategy", "head-tail")

In [ ]:
run("colab-head-weighted", "encoder", *GPU_ARGS, "--weighted")

In [ ]:
encoder_models = [RUNS / name for name in ("colab-head", "colab-head-tail", "colab-head-weighted")
                  if (RUNS / name / "metrics.json").is_file()]
if not encoder_models:
    raise ValueError("Run an encoder cell first, or use the CPU submission above.")
name = "final-colab-" + "-".join(path.name.removeprefix("colab-") for path in encoder_models)
final = run(name, "select", *cpu_models, *encoder_models)
print(final / "submission_1.csv")